> **Notebook-first lesson.** Examples are executable. Download-dependent examples are guarded so the notebook can still run offline.

## Mathematical Framework

Math companions for this lesson:

- [Math 00 · Notation & Shapes](../../math/00_notation_shapes.ipynb)
- [Math 02 · Calculus & Matrix Calculus](../../math/02_calculus_matrix_calculus.ipynb)
- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)
- [Math 11 · Attention & Transformer Mathematics](../../math/11_attention_transformers.ipynb)

Do not stop at the API surface. Identify the **spaces/vectors involved, objective or probability model, local derivatives, matrix shapes, and approximation assumptions**.

# Lesson 39: Build a Transformer block

A Transformer block combines:
- multi-head self-attention
- residual connections
- normalization
- feed-forward network

## Skeleton


In [ ]:
import torch
from torch import nn

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, attn_mask=None):
        a, _ = self.attn(x, x, x, attn_mask=attn_mask)
        x = self.norm1(x + a)
        x = self.norm2(x + self.ff(x))
        return x



## Residual connections
A residual path lets a block learn a correction to its input rather than rebuilding the whole representation.

## Feed-forward layer
Attention mixes information across positions. The feed-forward network performs nonlinear feature transformation independently at each position.

## Exercise
Implement the attention component yourself from Lesson 37/38 and swap it into this block.


## Runnable activity
This is a reduced-scale experiment for the core mechanism. Run it first, then extend it.

In [ ]:
import torch
from torch import nn
class Block(nn.Module):
    def __init__(self,d=16,h=4,ff=32):
        super().__init__()
        self.attn=nn.MultiheadAttention(d,h,batch_first=True)
        self.n1=nn.LayerNorm(d); self.n2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,ff),nn.GELU(),nn.Linear(ff,d))
    def forward(self,x):
        a,_=self.attn(x,x,x)
        x=self.n1(x+a)
        return self.n2(x+self.ff(x))
x=torch.randn(3,7,16); b=Block()
y=b(x)
print("input",x.shape,"output",y.shape)
y.mean().backward()
print("gradient reached first parameter:",next(b.parameters()).grad is not None)

## Explanation checkpoint
Explain the mechanism, the scale gap between this activity and production/research systems, and one experiment you would run next.